In [ ]:
# Make this notebook runnable from any working directory: locate the repository
# root by its marker file, then work from this notebook's own folder, which is
# what the relative paths below assume.
import os
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "requirements.txt").is_file() and _root != _root.parent:
    _root = _root.parent
_here = _root / "simulated-experiment"
if not _here.is_dir():
    raise RuntimeError(
        "Could not locate " + str(_here) + ". Run this notebook from inside the "
        "cloned repository."
    )
os.chdir(_here)

# Imports

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
import os
import matplotlib.pyplot as plt
import time
from smolagents import ToolCallingAgent, LiteLLMModel, PromptTemplates, tool, ActionStep, TaskStep, MessageRole

from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

from pro_russian_agents_simulated_exp import *
from vanilla_cn_agent import *
from refined_pro_ukrainian_agents import *

<path>//ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Creation of CNs (Run Once)

In [2]:
claims = pd.read_excel(r"../Data/Pro Russian top users and narratives.xlsx", sheet_name="top_20_narratives_20250313_1716")['description'].tolist()
claims

["NATO's eastward expansion threatened Russia's security and forced military intervention in Ukraine.",
 "Ukraine's government and military are portrayed as Nazi-controlled, requiring Russian 'denazification'.",
 "Russia's actions are a defensive response against NATO and Western geopolitical threats.",
 'The US and NATO sponsored a coup in Ukraine to install a pro-Western government.',
 'Russia is winning the war, inflicting heavy losses and liberating Ukrainian territories.',
 'Russia intervened to protect Russian-speaking populations from alleged Ukrainian government persecution.',
 'NATO and Western military-industrial complex intentionally prolong the Ukraine conflict to profit from it',
 "Crimea's referendum to join Russia was democratic and should be internationally recognized.",
 'US uses NGOs and protests to orchestrate regime change in targeted countries.',
 'Ukrainian government oppresses and discriminates against ethnic Russians and Russian speakers.',
 'Western media and g

In [4]:
print("\n".join(claims))

NATO's eastward expansion threatened Russia's security and forced military intervention in Ukraine.
Ukraine's government and military are portrayed as Nazi-controlled, requiring Russian 'denazification'.
Russia's actions are a defensive response against NATO and Western geopolitical threats.
The US and NATO sponsored a coup in Ukraine to install a pro-Western government.
Russia is winning the war, inflicting heavy losses and liberating Ukrainian territories.
Russia intervened to protect Russian-speaking populations from alleged Ukrainian government persecution.
NATO and Western military-industrial complex intentionally prolong the Ukraine conflict to profit from it
Crimea's referendum to join Russia was democratic and should be internationally recognized.
US uses NGOs and protests to orchestrate regime change in targeted countries.
Ukrainian government oppresses and discriminates against ethnic Russians and Russian speakers.
Western media and governments spread disinformation to demoni

## Using Refined CN Agents

In [ ]:
refined_agents = create_refined_pro_ukrainian_agents()
save_refined_pro_ukrainian_agents(refined_agents)

In [ ]:
help_dict = {
    "agent_1": "Persuasiveness",
    "agent_2": "Emotional Engagement",
    "agent_3": "Shareability (likeliness to be shared)"
}

In [ ]:
results_df = pd.DataFrame(columns=['claim_index', 'claim', 'cn_index', 'cn', 'kpi'])

for i in range(1, len(claims) + 1):
    claim = claims[i - 1]
    print(f"************** CLAIM {i} **************")
    for agent_name, agent in refined_agents[f"claim{i}"].items():
        kpi = help_dict[agent_name]
        history_path = os.path.join(REFINED_AGENTS_DIR, f"claim{i}", agent_name, "history.pkl")
        if os.path.exists(history_path):
            with open(history_path, 'rb') as f:
                history_list = pickle.load(f)
        else:
            history_list = []

        for j in range(3):
            if history_list == []:
                input = f"Claim: {claim}"
            else:
                prev_cns = "\n".join([f"- {cn}" for cn in history_list])
                input = f"Claim: {claim}\nPrevious Counter-Narratives:\n{prev_cns}"

            cn = agent.run(input, reset=False)
            history_list.append(cn)
            time.sleep(2)
            results_df.loc[len(results_df)] = [i, claim, j + 1, cn, kpi]
    
        with open(history_path, 'wb') as f:
            pickle.dump(history_list, f)

    save_refined_pro_ukrainian_agents_memories(refined_agents)

results_df.to_csv("simulated_exp_refined_treatment_cns.csv", index=False)

## Using Vanilla CN Agent

In [ ]:
vanilla_cn_agent = create_vanilla_cn_agent(system_prompt_agent_4, description_agent_4)
vanilla_cn_agent.save("Vanilla_CN_Agent")

kpis = ["Persuasiveness", "Emotional Engagement", "Shareability (likeliness to be shared)"]

In [ ]:
results_df_vanilla = pd.DataFrame(columns=['claim_index', 'claim', 'cn_index', 'cn', 'kpi'])

for i in range(1, len(claims) + 1):
    claim = claims[i - 1]
    print(f"************** CLAIM {i} **************")
    history_path = f"Vanilla_CN_Agent/claim{i}_history.pkl"
    if os.path.exists(history_path):
        with open(history_path, 'rb') as f:
            history_list = pickle.load(f)
    else:
        history_list = []

    for j in range(3):
        if history_list == []:
            input = f"Claim: {claim}"
        else:
            prev_cns = "\n".join([f"- {cn}" for cn in history_list])
            input = f"Claim: {claim}\nPrevious Counter-Narratives:\n{prev_cns}"

        cn = vanilla_cn_agent.run(input, reset=False)
        history_list.append(cn)
        time.sleep(2)
        for kpi in kpis:
            results_df_vanilla.loc[len(results_df_vanilla)] = [i, claim, j + 1, cn, kpi]

    # Save the history for the vanilla CN agent
    with open(history_path, 'wb') as f:
        pickle.dump(history_list, f)

    vanilla_cn_agent.memory.reset()

results_df_vanilla.to_csv("simulated_exp_vanilla_treatment_cns.csv", index=False)

# Control Condition

In [ ]:
type_1_evaluator_agents = create_pro_russian_agents_simulated_exp(pd.read_excel(r"../Data/pro_russian_users_data_for_agents.xlsx"), 1)
save_pro_russian_agents(type_1_evaluator_agents, 1)

claims = pd.read_excel(r"../Data/Pro Russian top users and narratives.xlsx", sheet_name="top_20_narratives_20250313_1716")['description'].tolist()
kpis = ["Persuasiveness", "Emotional Engagement", "Shareability (likeliness to be shared)"]

In [ ]:
control_results_df = pd.DataFrame(columns=['claim_index', 'claim', 'kpi', 'evaluator_agent_name', 'score'])

for i in range(1, len(claims) + 1):
    claim = claims[i - 1]
    for kpi in kpis:
        for agent_name, agent in type_1_evaluator_agents.items():
            input = f"""
pro-Russian claim: {claim}
KPI: {kpi}
    """
            score = agent.run(input)
            control_results_df.loc[len(control_results_df)] = [i, claim, kpi, agent_name, score]
            time.sleep(2)
    
    control_results_df.to_csv("simulated_experiment_results/control_condition.csv", index=False)

# Vanilla Treatment Condition

In [ ]:
type_2_evaluator_agents = create_pro_russian_agents_simulated_exp(pd.read_excel(r"../Data/pro_russian_users_data_for_agents.xlsx"), 2)
save_pro_russian_agents(type_2_evaluator_agents, 2)

vanilla_cns = pd.read_csv("simulated_exp_vanilla_treatment_cns.csv", encoding='utf-8')

In [ ]:
vanilla_treatment_results_df = pd.DataFrame(columns=['claim_index', 'claim', 'cn_index', 'cn', 'kpi', 'evaluator_agent_name', 'score'])

for idx, row in vanilla_cns.iterrows():
    claim_index = row['claim_index']
    claim = row['claim']
    cn_index = row['cn_index']
    cn = row['cn']
    kpi = row['kpi']
    
    for agent_name, agent in type_2_evaluator_agents.items():
        input = f"""
pro-Russian claim: {claim}
pro-Ukrainian Counter-Narrative: {cn}
KPI: {kpi}
"""
        score = agent.run(input)
        vanilla_treatment_results_df.loc[len(vanilla_treatment_results_df)] = [claim_index, claim, cn_index, cn, kpi, agent_name, score]
        time.sleep(2)

    vanilla_treatment_results_df.to_csv("simulated_experiment_results/vanilla_treatment_condition.csv", index=False)

# Refined Treatment Condition

In [ ]:
type_2_evaluator_agents = create_pro_russian_agents_simulated_exp(pd.read_excel(r"../Data/pro_russian_users_data_for_agents.xlsx"), 2)
save_pro_russian_agents(type_2_evaluator_agents, 2)

refined_cns = pd.read_csv("simulated_exp_refined_treatment_cns.csv", encoding='utf-8', encoding_errors="ignore")
refined_cns.head()

In [ ]:
refined_treatment_results_df = pd.DataFrame(columns=['claim_index', 'claim', 'cn_index', 'cn', 'kpi', 'evaluator_agent_name', 'score'])

for idx, row in refined_cns.iterrows():
    claim_index = row['claim_index']
    claim = row['claim']
    cn_index = row['cn_index']
    cn = row['cn']
    kpi = row['kpi']
    
    for agent_name, agent in type_2_evaluator_agents.items():
        input = f"""
pro-Russian claim: {claim}
pro-Ukrainian Counter-Narrative: {cn}
KPI: {kpi}
"""
        score = agent.run(input)
        refined_treatment_results_df.loc[len(refined_treatment_results_df)] = [claim_index, claim, cn_index, cn, kpi, agent_name, score]
        time.sleep(2)

    refined_treatment_results_df.to_csv("simulated_experiment_results/refined_treatment_condition.csv", index=False)

# Results Analysis

## Visualization

In [ ]:
control = pd.read_csv("simulated_experiment_results/control_condition.csv")
vanilla = pd.read_csv("simulated_experiment_results/vanilla_treatment_condition.csv")
refined = pd.read_csv("simulated_experiment_results/refined_treatment_condition.csv")

# 2. Add a 'condition' column to each
control['condition'] = 'Control'
vanilla['condition'] = 'Vanilla'
refined['condition'] = 'Refined'

df_all = pd.concat([control, vanilla, refined], ignore_index=True)

# 4. Plot for each KPI
for kpi in df_all['kpi'].unique():
    subset = df_all[df_all['kpi'] == kpi]
    scores_by_cond = [
        subset[subset['condition'] == cond]['score'].values
        for cond in ['Control', 'Vanilla', 'Refined']
    ]
    
    # Boxplot
    fig, ax = plt.subplots()
    ax.boxplot(scores_by_cond, labels=['Control', 'Vanilla', 'Refined'])
    ax.set_title(f"Boxplot of Scores for KPI: {kpi}")
    ax.set_ylabel("Score")
    plt.show()
    
    # Violin Plot
    fig, ax = plt.subplots()
    ax.violinplot(scores_by_cond, showmeans=True)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['Control', 'Vanilla', 'Refined'])
    ax.set_title(f"Violin Plot of Scores for KPI: {kpi}")
    ax.set_ylabel("Score")
    plt.show()

## Kruskal-Wallis + Dunn

In [ ]:
from scipy.stats import kruskal
import scikit_posthocs as sp

control = pd.read_csv("simulated_experiment_results/control_condition.csv")
vanilla = pd.read_csv("simulated_experiment_results/vanilla_treatment_condition.csv")
refined = pd.read_csv("simulated_experiment_results/refined_treatment_condition.csv")

control['condition'] = 'Control'
vanilla['condition'] = 'Vanilla'
refined['condition'] = 'Refined'

df_all = pd.concat([control, vanilla, refined], ignore_index=True)

for kpi in df_all['kpi'].unique():
    sub = df_all[df_all['kpi'] == kpi]
    groups = [sub[sub['condition']==c]['score'] 
              for c in ['Control','Vanilla','Refined']]
    
    # Kruskal–Wallis global test
    H, p_global = kruskal(*groups)
    print(f"\nKPI: {kpi}")
    print(f"  Kruskal–Wallis H = {H:.3f}, p = {p_global:.3f}")
    
    if p_global < 0.05:
        # Dunn’s post-hoc with Bonferroni correction
        # Define contrasts of interest
        contrasts = [('Control', 'Vanilla'), ('Control', 'Refined'), ('Vanilla', 'Refined')]
        # Perform Dunn’s test with Bonferroni correction
        dunn = sp.posthoc_dunn(sub, val_col='score', group_col='condition', p_adjust='bonferroni')
        
        # Extract only the contrasts of interest
        results = []
        for a, b in contrasts:
            results.append({
                'Comparison': f'{a} vs {b}',
                'p-value': dunn.loc[a, b]
            })
        df_results = pd.DataFrame(results)
        print(df_results)


KPI: Persuasiveness
  Kruskal–Wallis H = 186.045, p = 0.000
           Comparison       p-value
0  Control vs Vanilla  1.142308e-27
1  Control vs Refined  1.484773e-41
2  Vanilla vs Refined  7.728271e-04

KPI: Emotional Engagement
  Kruskal–Wallis H = 466.944, p = 0.000
           Comparison       p-value
0  Control vs Vanilla  5.740172e-24
1  Control vs Refined  1.027668e-90
2  Vanilla vs Refined  2.366120e-45

KPI: Shareability (likeliness to be shared)
  Kruskal–Wallis H = 509.564, p = 0.000
           Comparison        p-value
0  Control vs Vanilla   4.020919e-39
1  Control vs Refined  8.015482e-107
2  Vanilla vs Refined   2.382363e-35


## T-test and Cohen’s d

In [ ]:
control_results_df = pd.read_csv("simulated_experiment_results/control_condition.csv")
vanilla_treatment_results_df = pd.read_csv("simulated_experiment_results/vanilla_treatment_condition.csv")
refined_treatment_results_df = pd.read_csv("simulated_experiment_results/refined_treatment_condition.csv")

In [4]:
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    dof = nx + ny - 2
    pooled_var = ((nx - 1)*x.var(ddof=1) + (ny - 1)*y.var(ddof=1)) / dof
    return (x.mean() - y.mean()) / np.sqrt(pooled_var)

for kpi in control_results_df['kpi'].unique():
    print(f"\n=== KPI: {kpi} ===")
    ctrl_scores  = control_results_df[ control_results_df['kpi'] == kpi ]['score']
    van_scores   = vanilla_treatment_results_df[ vanilla_treatment_results_df['kpi'] == kpi ]['score']
    ref_scores   = refined_treatment_results_df[ refined_treatment_results_df['kpi'] == kpi ]['score']
    
    # Compute and print means
    means = {
        'Control': float(ctrl_scores.mean()),
        'Vanilla': float(van_scores.mean()),
        'Refined': float(ref_scores.mean())
    }
    print("Means:", means)
    
    # Contrast definitions
    contrasts = [
        ('Control', ctrl_scores, 'Vanilla', van_scores),
        ('Control', ctrl_scores, 'Refined', ref_scores),
        ('Vanilla', van_scores, 'Refined', ref_scores),
    ]
    
    # Run t-tests + effect sizes
    for name1, scores1, name2, scores2 in contrasts:
        t_stat, p_val = ttest_ind(scores1, scores2, equal_var=False)
        d = cohens_d(scores1, scores2)
        print(f"{name1} vs. {name2}: p = {p_val}, Cohen's d = {d}")


=== KPI: Persuasiveness ===
Means: {'Control': 91.38888888888889, 'Vanilla': 56.9537037037037, 'Refined': 49.8287037037037}
Control vs. Vanilla: p = 1.3892282710860793e-119, Cohen's d = 1.0110811751713915
Control vs. Refined: p = 7.18714146148364e-149, Cohen's d = 1.1452984117526754
Vanilla vs. Refined: p = 3.618141101722519e-05, Cohen's d = 0.17812703790273965

=== KPI: Emotional Engagement ===
Means: {'Control': 87.5, 'Vanilla': 56.0787037037037, 'Refined': 33.27777777777778}
Control vs. Vanilla: p = 6.31443268330252e-91, Cohen's d = 0.941905078554917
Control vs. Refined: p = 9.172247111332561e-216, Cohen's d = 1.658248694815993
Vanilla vs. Refined: p = 9.958305255587803e-45, Cohen's d = 0.6179522198826983

=== KPI: Shareability (likeliness to be shared) ===
Means: {'Control': 87.80555555555556, 'Vanilla': 58.93055555555556, 'Refined': 39.18518518518518}
Control vs. Vanilla: p = 1.21384980938929e-91, Cohen's d = 0.9069302479255908
Control vs. Refined: p = 1.42906132935108e-226, Cohe

## Mixed-Effects Modeling
### A mixed-effects (hierarchical) model lets you:
### - Treat condition (Control/Vanilla/Refined) as a fixed effect you care about.
### - Treat evaluator_agent_name as a random effect, capturing agent-to-agent variability.

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

control = pd.read_csv("simulated_experiment_results/control_condition.csv")
vanilla = pd.read_csv("simulated_experiment_results/vanilla_treatment_condition.csv")
refined = pd.read_csv("simulated_experiment_results/refined_treatment_condition.csv")

control['condition'] = 'Control'
vanilla['condition'] = 'Vanilla'
refined['condition'] = 'Refined'

df_all = pd.concat([control, vanilla, refined], ignore_index=True)

# 4. Fit MixedLM: score ~ condition + (1 | evaluator_agent_name)
model = smf.mixedlm(
    "score ~ C(condition)", 
    data=df_all, 
    groups=df_all["evaluator_agent_name"]
)
result = model.fit()
print(result.summary())